# Computation of observation time distribution in direction

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import ROOT, glob

plt.rcParams['figure.dpi'] = 400
plt.rcParams['savefig.dpi'] = 400

In [ ]:
from astropy.coordinates import SkyCoord
target_l = 0
target_b = 0
target_gal = SkyCoord(l=target_l, b=target_b, unit='deg', frame='galactic')

In [ ]:
dirDir= "/home/users/l/lebrun/"
resultPath = dirDir + "results/"
dataPath = "/dpnc/beegfs/dampe/prod/FM/skim/6.0.0/v2/2016/01/"
filenames = [dataPath + "*_data_photon.root"]

for filename in filenames:
    # Get all files matching the pattern
    # Use glob to find all files that match the pattern
    files = sorted(glob.glob(filename))
print(files)
treename='HousekeepingData/SatStatus'

rdf_ = ROOT.RDataFrame(treename, files)
rdf_.GetColumnNames()


In [ ]:
# print("Created RDataFrame")
# df = rdf.AsNumpy(['DmpHKDSatStatus.fSecond', 'DmpHKDSatStatus.fMicrosecond',
#                    'DmpHKDSatStatus.fPosition.fX', 
#                    'DmpHKDSatStatus.fPosition.fY', 
#                    'DmpHKDSatStatus.fPosition.fZ'])
# print("Converted to numpy")
# df = pd.DataFrame(df)
# df.rename(columns={'DmpHKDSatStatus.fSecond': 'second', 
#                    'DmpHKDSatStatus.fMicrosecond': 'microsecond',
#                    'DmpHKDSatStatus.fPosition.fX': 'x',
#                    'DmpHKDSatStatus.fPosition.fY': 'y',
#                    'DmpHKDSatStatus.fPosition.fZ': 'z'}, inplace=True)
# print("Converted to pandas")

In [ ]:
from astropy.time import Time, TimeDelta
import numpy as np

def met_to_utc(seconds, microseconds):
    seconds = np.asarray(seconds)
    microseconds = np.asarray(microseconds)
    delta_sec = TimeDelta(seconds, format='sec')
    delta_us = TimeDelta(microseconds * 1e-6, format='sec')
    epoch = Time("2013-01-01T00:00:00", scale="utc")
    return epoch + delta_sec + delta_us

def met_to_utc_string(seconds, microseconds):
    # Ensure input is numpy array
    full_time = met_to_utc(seconds, microseconds)
    # Return human-readable UTC string
    return full_time.utc.iso


### Eliminate duplicates

In [ ]:
# # Print the number of entries
# print("Number of entries:", len(df['second']))
# df = df.sort_values(by=['second', 'microsecond']).drop_duplicates('second', keep='first')
# # UTC filtering to ensure no overlap between days
# utc_time = met_to_utc(df['second'], df['microsecond'])
# mask = (utc_time >= day_start) & (utc_time < day_end)
# df = df.loc[mask].reset_index(drop=True)
# df['utc'] = utc_time[mask]
# # Print the number of entries with Delta_t != NaN
# print("Number of entries w/o duplicates:", len(df['second']))
# print(f"Timestamps span from {met_to_utc_string(df.iloc[0]['second'], df.iloc[0]['microsecond'])} to {met_to_utc_string(df.iloc[-1]['second'], df.iloc[-1]['microsecond'])}")

In [ ]:
def load_day_df(file, day_start_str, verbose=True):

    day_start = Time(f"{day_start_str}T00:00:00", scale='utc')
    day_end = day_start + TimeDelta(1, format='jd') 

    rdf = ROOT.RDataFrame(treename, file)
    if verbose : print("Created RDataFrame")
    df = rdf.AsNumpy(['DmpHKDSatStatus.fSecond', 'DmpHKDSatStatus.fMicrosecond',
                    'DmpHKDSatStatus.fPosition.fX', 
                    'DmpHKDSatStatus.fPosition.fY', 
                    'DmpHKDSatStatus.fPosition.fZ',
                    'DmpHKDSatStatus.fAngle'])
    if verbose : print("Converted to numpy")
    df = pd.DataFrame(df)

    df.rename(columns={'DmpHKDSatStatus.fSecond': 'second', 
                    'DmpHKDSatStatus.fMicrosecond': 'microsecond',
                    'DmpHKDSatStatus.fPosition.fX': 'x',
                    'DmpHKDSatStatus.fPosition.fY': 'y',
                    'DmpHKDSatStatus.fPosition.fZ': 'z',
                    'DmpHKDSatStatus.fAngle' : 'fAngle'}, inplace=True)
    if verbose : print("Converted to pandas")
    # Print the number of entries
    if verbose : print("Number of entries:", len(df['second']))
    df = df.sort_values(by=['second', 'microsecond']).drop_duplicates('second', keep='first')
    # UTC filtering to ensure no overlap between days
    utc_time = met_to_utc(df['second'], df['microsecond'])
    mask = (utc_time >= day_start) & (utc_time < day_end)
    df = df.loc[mask].reset_index(drop=True)
    df['utc'] = utc_time[mask]
    # Print the number of entries with Delta_t != NaN
    if verbose : print("Number of entries w/o duplicates:", len(df['second']))
    if verbose : print(f"Timestamps span from {met_to_utc_string(df.iloc[0]['second'], df.iloc[0]['microsecond'])} to {met_to_utc_string(df.iloc[-1]['second'], df.iloc[-1]['microsecond'])}")

    return df
        

### SAA

In [ ]:
from shapely.geometry import Point, Polygon
from shapely import vectorized
# SAA Region

saa_lat_2016 = [
    -8.136145, -5.522644, -2.909143, -0.295642, 2.317859, 4.931360,
    -1.170294, -5.924736, -10.080192, -13.636662, -16.594145, -18.952644,
    -32.560543, -34.555897, -36.591114, -38.661793, -40.743484, -42.791676,
    -44.741795, -46.509228, -47.989288, -49.057251, -49.568325, -49.357666,
    -48.240376, -46.011497, -42.446022, -37.298885, -30.304964, -21.179083
]

saa_lon_2016 = [
    -95.0, -84.0, -73.0, -62.0, -51.0, -40.0,
    -35.0, -29.0, -23.0, -17.0, -11.0, -5.0,
    -5.0, -10.0, -15.0, -20.0, -25.0, -30.0,
    -35.0, -40.0, -45.0, -50.0, -55.0, -60.0,
    -65.0, -70.0, -75.0, -80.0, -85.0, -90.0
]

saa_polygon_2016 = list(zip(saa_lon_2016, saa_lat_2016))
saa_polygon_shape = Polygon(saa_polygon_2016)  # Define SAA region


In [ ]:
from astropy.coordinates import CartesianRepresentation, SkyCoord, angular_separation, EarthLocation, GCRS, ITRS
from astropy.coordinates import Longitude, Latitude, AltAz
import astropy.units as u
def mask_saa(df):
    """
    Mask out timestamps where the satellite is within the SAA region.
    
    Parameters:
    - df: pandas.DataFrame with columns ['second', 'microsecond', 'x', 'y', 'z', 'utc']
    
    Returns:
    - df_masked: DataFrame excluding SAA timestamps
    - mask: boolean mask array (True = keep, False = SAA)
    """
    # Convert MET to UTC Time objects
    obstime = df['utc']

    # Cartesian coordinates in meters
    positions = CartesianRepresentation(
        x=df['x'].to_numpy() * u.m,
        y=df['y'].to_numpy() * u.m,
        z=df['z'].to_numpy() * u.m
    )

    # Build GCRS coordinate frame at each time
    gcrs_coords = GCRS(positions, obstime=obstime)

    # Convert to Earth-fixed frame (ITRS), then to geographic coordinates
    itrs_coords = gcrs_coords.transform_to(ITRS(obstime=obstime))
    geodetic = itrs_coords.earth_location.to_geodetic()

    # Extract lat/lon in degrees
    lat = geodetic.lat.to(u.deg).value
    lon = geodetic.lon.to(u.deg).value

    # Define SAA region
    mask_inside_saa = vectorized.contains(saa_polygon_shape, lon, lat)

    # Invert to create a mask of valid points
    valid_mask = ~mask_inside_saa
    df_masked = df[valid_mask].reset_index(drop=True)
    return df_masked, valid_mask

### Create bins

In [ ]:
cos_theta_min = 0.5
cos_theta_max = 1
n_theta_bins = 10

phi_min = 0
phi_max = 360
n_phi_bins = 20

theta_bins_edge = np.linspace(cos_theta_min, cos_theta_max, n_theta_bins + 1)
theta_bins_center = (theta_bins_edge[:-1] + theta_bins_edge[1:])/2

phi_bins_edge = np.linspace(phi_min, phi_max, n_phi_bins + 1)
phi_bins_center = (phi_bins_edge[:-1] + phi_bins_edge[1:])/2

### Process data

In [ ]:
import re

def extract_date(path):
    match = re.search(r'/(\d{4})/(\d{2})/(\d{2})_', path)
    if match:
        year, month, day = match.groups()
        return f"{year}-{month}-{day}"
    return None

def get_cos_theta(df) :
     # 1. Convert position vectors
    positions = CartesianRepresentation(
        x=df['x'].values * u.m,
        y=df['y'].values * u.m,
        z=df['z'].values * u.m
    )

    obstime = df['utc']
    zenith_coords = GCRS(positions, obstime=obstime)

    # 2. Transform target into GCRS frame at each obstime
    target_gcrs = target_gal.transform_to(GCRS(obstime=obstime))

    # 3. Extract unit vectors for both zenith and target
    z_unit = zenith_coords.cartesian.get_xyz().value.T
    z_unit /= np.linalg.norm(z_unit, axis=1, keepdims=True)

    t_unit = target_gcrs.cartesian.get_xyz().value.T
    t_unit /= np.linalg.norm(t_unit, axis=1, keepdims=True)

    # 4. Dot product = cos(theta)
    cos_theta = np.einsum('ij,ij->i', z_unit, t_unit)
    return cos_theta


In [ ]:
file = files[0]
start_date = extract_date(file)
df = load_day_df(file, start_date, verbose=False)
v = df['fAngle'][20]
print(v.X())
print(v.Y())
print(v.Z())

In [ ]:

# For each day, load df and process

histogram = np.zeros(len(theta_bins_edge) - 1)

for file in files :
    print(f"file : ", file)
    start_date = extract_date(file)
    print(start_date)
    df = load_day_df(file, start_date)
    df, valid_mask = mask_saa(df)

    cos_theta = get_cos_theta(df)

    # Create a histogram for this day
    daily_hist, _ = np.histogram(cos_theta, bins=theta_bins_edge)
    print(daily_hist)

    # Accumulate into the total histogram
    histogram += daily_hist

    

### Load bins

In [ ]:
# PATH_BINS = "/home/users/l/lebrun/results/effArea_bins_info_HET.npz"
# # Load the effective area table
# EFF_AREA_BINS = np.load(PATH_BINS)

## Get bins edges and centers
# ENERGY_BINS_EDGES = EFF_AREA_BINS['E_bins_edge_HET']
# ENERGY_BINS_CENTERS = EFF_AREA_BINS['E_bins_center_HET']
# COSTHETA_BINS_EDGES = EFF_AREA_BINS['theta_bins_edge']
# COSTHETA_BINS_CENTERS = EFF_AREA_BINS['theta_bins_center']

In [ ]:
plt.figure(figsize=(8, 5))
plt.step(theta_bins_center, histogram, where='mid', color='blue', label='observing time')

plt.xlabel(r'$\cos(\theta)$')
plt.ylabel('Counts (or Exposure)')
plt.title('Observation Distribution vs. $\cos(\\theta)$')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()